# Campaign ROI Optimization Engine
## Bank Marketing — Profit-Driven Customer Targeting

> **Research Notebook** | Data Understanding → Feature Engineering → Model Comparison → Profit Optimization

---

### Business Problem
A Portuguese bank runs telephone marketing campaigns to sell term deposits.  
**Goal:** Given a fixed daily call capacity, rank customers by *expected profit* so the campaign team contacts the most valuable customers first.

**Expected Profit** = P(subscribe) × Revenue − Cost

This transforms a binary classification problem into a **decision optimisation** problem under operational constraints.

---

### Dataset
[UCI Bank Marketing Dataset](https://archive.ics.uci.edu/dataset/222/bank+marketing) — 41,188 rows, 20 features.  
Target: `y` — did the client subscribe a term deposit? (heavily imbalanced: ~89% No)


In [ ]:
# ── Standard imports ──────────────────────────────────────────────────────────
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Make src/ importable from the notebook
sys.path.insert(0, str(Path().resolve().parent / 'src'))

from bank_roi.config import cfg
from bank_roi.data import load_raw, engineer_features, split
from bank_roi.models import build_pipeline, all_pipelines
from bank_roi.evaluation import compare_models_cv, holdout_metrics, summary_table
from bank_roi.optimization import score_customers, profit_at_capacity, profit_curve

%matplotlib inline
sns.set_theme(style='whitegrid', palette='Blues_r')
pd.set_option('display.max_columns', 30)
print('Setup complete ✓')

## 1. Data Loading & Exploration

In [ ]:
df_raw = load_raw()
print(f'Shape: {df_raw.shape}')
df_raw.head(3)

In [ ]:
# Class imbalance — critical for model selection and threshold decisions
target_dist = df_raw['y'].value_counts(normalize=True)
print('Target distribution:')
print(target_dist.to_string())
print(f'\nImbalance ratio: {target_dist["no"]/target_dist["yes"]:.1f}:1 (no:yes)')

In [ ]:
# Numeric summary
df_raw.describe().T.style.background_gradient(cmap='Blues', axis=1)

In [ ]:
# Categorical cardinality
cat_cols = df_raw.select_dtypes('object').columns
print('Categorical columns and unique values:')
df_raw[cat_cols].nunique().sort_values(ascending=False)

## 2. Feature Engineering

Two key decisions:
1. **Drop `duration`** — it measures how long the call lasted, which is only known *after* the call. Including it would cause target leakage at inference time.
2. **Engineer `previous_contacted`** — `pdays=999` means the customer was never previously contacted. We convert this sentinel to a binary flag and drop the original column.

In [ ]:
df = engineer_features(df_raw)
print(f'After engineering: {df.shape}')
print(f'previous_contacted: {df["previous_contacted"].value_counts().to_dict()}')
df.head(3)

In [ ]:
X_train, X_test, y_train, y_test = split(df)
print(f'Train: {X_train.shape} | Positive rate: {y_train.mean():.2%}')
print(f'Test:  {X_test.shape} | Positive rate: {y_test.mean():.2%}')

## 3. Model Comparison — 5-Fold Stratified Cross-Validation

Three candidate models evaluated on:
- **ROC-AUC** — discrimination ability (threshold-independent)
- **PR-AUC** (Average Precision) — precision/recall trade-off, more informative under class imbalance

All models use a `ColumnTransformer` preprocessing step (OneHotEncoder + StandardScaler) inside a `Pipeline` to prevent leakage across folds.

In [ ]:
%%time
pipelines = all_pipelines(X_train)
cv_results = compare_models_cv(pipelines, X_train, y_train)

In [ ]:
table = summary_table(cv_results)
print('=== 5-Fold CV Results (mean ± std) ===')
display(table.style.highlight_max(subset=['roc_auc_mean', 'pr_auc_mean'], color='#d4edda'))

In [ ]:
# Visualise fold-level variance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric, title in zip(
    axes,
    ['roc_auc_val', 'pr_auc_val'],
    ['ROC-AUC (validation)', 'PR-AUC (validation)'],
):
    sns.boxplot(data=cv_results, x='model', y=metric, ax=ax, palette='Blues')
    ax.set_title(title)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig('../outputs/cv_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Best Model — Hold-Out Evaluation

In [ ]:
best_name = cv_results.groupby('model')['roc_auc_val'].mean().idxmax()
print(f'Best model by CV ROC-AUC: {best_name}')

best_pipeline = pipelines[best_name]
best_pipeline.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay

metrics = holdout_metrics(best_pipeline, X_test, y_test)
print(f"ROC-AUC     : {metrics['roc_auc']:.4f}")
print(f"PR-AUC      : {metrics['pr_auc']:.4f}")
print(f"Brier Score : {metrics['brier_score']:.4f}  (lower = better calibration)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
RocCurveDisplay.from_estimator(best_pipeline, X_test, y_test, ax=axes[0], name=best_name)
PrecisionRecallDisplay.from_estimator(best_pipeline, X_test, y_test, ax=axes[1], name=best_name)
plt.tight_layout()
plt.savefig('../outputs/holdout_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Profit Optimisation

Given call capacity N, we want to maximise:

$$\text{Profit}(N) = \sum_{i=1}^{N} \left[ p_i \times \text{Revenue} - \text{Cost} \right]$$

where customers are sorted by expected profit descending.

In [ ]:
X_all = df.drop(columns=['y'])
scored = score_customers(best_pipeline, X_all)

print('Scored customers sample:')
scored[['rank', 'p_subscribe', 'expected_profit', 'cum_profit', 'cum_roi']].head(10)

In [ ]:
# Capacity sensitivity analysis
capacities = [1000, 2500, 5000, 10000, 20000, len(scored)]
kpi_table = pd.DataFrame([profit_at_capacity(scored, c) for c in capacities])
kpi_table['roi'] = kpi_table['roi'].map('{:.1%}'.format)
kpi_table['total_expected_profit'] = kpi_table['total_expected_profit'].map('£{:,.0f}'.format)
kpi_table['total_cost'] = kpi_table['total_cost'].map('£{:,.0f}'.format)
display(kpi_table)

In [ ]:
pc = profit_curve(scored)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=pc['rank'], y=pc['cum_profit'],
    mode='lines', name='Cumulative Profit',
    line=dict(color='royalblue', width=2),
    fill='tozeroy', fillcolor='rgba(65,105,225,0.1)'
))
fig.add_vline(x=5000, line_dash='dash', line_color='red',
              annotation_text='5,000 calls', annotation_position='top right')
fig.update_layout(
    title='Cumulative Expected Profit by Call Rank',
    xaxis_title='Customers (ranked by expected profit)',
    yaxis_title='Cumulative Expected Profit (£)',
    height=400,
)
fig.show()

## 6. Export Artifacts
Outputs consumed by the Streamlit dashboard.

In [ ]:
from pathlib import Path
import joblib

Path('../outputs/models').mkdir(parents=True, exist_ok=True)
joblib.dump(best_pipeline, f'../outputs/models/best_model_{best_name}.joblib')

scored.to_csv('../outputs/scored_customers.csv', index=False)
pc.to_csv('../outputs/profit_curve.csv', index=False)
cv_results.to_csv('../outputs/cv_results.csv', index=False)

print('Artifacts saved:')
print('  outputs/models/best_model_*.joblib')
print('  outputs/scored_customers.csv')
print('  outputs/profit_curve.csv')
print('  outputs/cv_results.csv')